[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ItsNotAILABS/PARALLAX-Exchange-Clearinghouse/blob/main/examples/03_mint_ai_token.ipynb)

# 03 — Mint an AI Token

This notebook demonstrates the **Token Factory** — minting AI tokens with phi-derived supply mechanics:
- Create custom AI token types (Compute, GPU, Agent, etc.)
- Mint tokens with Fibonacci × φ supply caps
- View token balances and holder distribution
- Auto-list minted tokens on the Phantom Exchange

In [ ]:
!pip install -q numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from enum import Enum

# PHI constants
PHI = 1.6180339887498948482
PHI_INV = 1.0 / PHI
PHI_SQ = PHI * PHI

def fibonacci(n: int) -> List[int]:
    seq = [1, 1]
    for i in range(2, n):
        seq.append(seq[-1] + seq[-2])
    return seq

FIB_21 = fibonacci(21)
print(f"φ² = {PHI_SQ:.6f}")
print(f"Fibonacci sequence: {FIB_21}")
print("✅ Ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TOKEN FACTORY — Sovereign Token Minting
# ═══════════════════════════════════════════════════════════════

class TokenCategory(Enum):
    CORE_INFRASTRUCTURE = "Core AI Infrastructure"
    ADVANCED_RESOURCES = "Advanced AI Resources"
    SPECIALIZED = "Specialized AI Capabilities"
    GOVERNANCE = "AI Governance"
    CREATOR = "Creator Tokens"

@dataclass
class TokenDefinition:
    token_code: str
    name: str
    category: TokenCategory
    fib_index: int
    supply_cap: int
    minted: float = 0.0
    holders: Dict[str, float] = field(default_factory=dict)
    created_at: float = field(default_factory=lambda: __import__('time').time())

@dataclass
class MintEvent:
    token_code: str
    recipient: str
    amount: float
    new_supply: float
    supply_cap: int
    utilization: float

class TokenFactory:
    """Sovereign Token Factory — phi-derived supply mechanics"""
    
    def __init__(self):
        self.tokens: Dict[str, TokenDefinition] = {}
        self.mint_events: List[MintEvent] = []
    
    def create_token(self, code: str, name: str, category: TokenCategory,
                     fib_index: int = 10) -> TokenDefinition:
        """Create a new AI token with Fibonacci × φ² supply cap"""
        supply_cap = int(FIB_21[min(fib_index, 20)] * PHI_SQ)
        token = TokenDefinition(
            token_code=code, name=name, category=category,
            fib_index=fib_index, supply_cap=supply_cap
        )
        self.tokens[code] = token
        return token
    
    def mint(self, code: str, recipient: str, amount: float) -> Optional[MintEvent]:
        """Mint tokens to a recipient. Returns None if supply cap exceeded."""
        token = self.tokens.get(code)
        if not token:
            print(f"❌ Token {code} does not exist")
            return None
        if token.minted + amount > token.supply_cap:
            print(f"❌ Mint would exceed supply cap ({token.minted + amount:.0f} > {token.supply_cap})")
            return None
        
        token.minted += amount
        token.holders[recipient] = token.holders.get(recipient, 0) + amount
        
        event = MintEvent(
            token_code=code, recipient=recipient, amount=amount,
            new_supply=token.minted, supply_cap=token.supply_cap,
            utilization=token.minted / token.supply_cap
        )
        self.mint_events.append(event)
        return event
    
    def get_summary_dataframe(self) -> pd.DataFrame:
        rows = []
        for code, t in self.tokens.items():
            rows.append({
                "Code": code, "Name": t.name, "Category": t.category.value,
                "Fib Index": t.fib_index, "Supply Cap": t.supply_cap,
                "Minted": t.minted, "Holders": len(t.holders),
                "Utilization": f"{(t.minted/t.supply_cap)*100:.1f}%"
            })
        return pd.DataFrame(rows)

print("✅ TokenFactory defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: Create AI token types
# ═══════════════════════════════════════════════════════════════

factory = TokenFactory()

# Core Infrastructure tokens
factory.create_token("AICPU", "AI Compute Token", TokenCategory.CORE_INFRASTRUCTURE, 15)
factory.create_token("AIMEM", "AI Memory Token", TokenCategory.CORE_INFRASTRUCTURE, 14)
factory.create_token("AIINF", "AI Inference Token", TokenCategory.CORE_INFRASTRUCTURE, 16)

# Advanced Resources
factory.create_token("AIGPU", "AI GPU Hours", TokenCategory.ADVANCED_RESOURCES, 12)
factory.create_token("AIAGENT", "AI Agent Execution", TokenCategory.ADVANCED_RESOURCES, 13)
factory.create_token("AIRAG", "AI RAG Queries", TokenCategory.ADVANCED_RESOURCES, 14)

# Specialized
factory.create_token("AIVIS", "AI Vision Processing", TokenCategory.SPECIALIZED, 11)
factory.create_token("AICODE", "AI Code Generation", TokenCategory.SPECIALIZED, 13)

print("TOKEN REGISTRY:")
print("=" * 70)
df = factory.get_summary_dataframe()
print(df[["Code", "Name", "Supply Cap"]].to_string(index=False))
print(f"\nSupply formula: Fibonacci[n] × φ² = Fib[n] × {PHI_SQ:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: Mint tokens to recipients
# ═══════════════════════════════════════════════════════════════

print("MINTING AI TOKENS:")
print("=" * 60)

mints = [
    ("AICPU", "research_lab_mit", 5000.0),
    ("AICPU", "startup_anthropic", 3000.0),
    ("AIGPU", "lab_deepmind", 200.0),
    ("AIGPU", "independent_researcher", 50.0),
    ("AIAGENT", "builder_alice", 100.0),
    ("AIAGENT", "builder_bob", 150.0),
    ("AIAGENT", "dao_treasury", 500.0),
    ("AIRAG", "enterprise_acme", 1000.0),
    ("AIVIS", "artist_collective", 75.0),
    ("AICODE", "open_source_fund", 300.0),
]

for code, recipient, amount in mints:
    event = factory.mint(code, recipient, amount)
    if event:
        print(f"  ✅ Minted {amount:>8,.0f} {code:<8} → {recipient}")
        print(f"     Supply: {event.new_supply:,.0f} / {event.supply_cap:,} ({event.utilization:.2%})")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: View token state and holder distribution
# ═══════════════════════════════════════════════════════════════

print("TOKEN SUMMARY:")
print("=" * 70)
print(factory.get_summary_dataframe().to_string(index=False))

# Show holder breakdown for AIAGENT
print(f"\n\nAIAGENT Holder Breakdown:")
print("-" * 40)
token = factory.tokens["AIAGENT"]
for holder, balance in sorted(token.holders.items(), key=lambda x: -x[1]):
    pct = (balance / token.minted) * 100
    bar = "█" * int(pct / 5)
    print(f"  {holder:<25} {balance:>8,.0f} ({pct:>5.1f}%) {bar}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: Visualize supply mechanics
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Supply caps by token (Fibonacci × φ²)
codes = [t.token_code for t in factory.tokens.values()]
caps = [t.supply_cap for t in factory.tokens.values()]
minted = [t.minted for t in factory.tokens.values()]

x = np.arange(len(codes))
axes[0].bar(x, caps, alpha=0.3, color='gold', label='Supply Cap (Fib×φ²)')
axes[0].bar(x, minted, alpha=0.8, color='blue', label='Minted')
axes[0].set_xticks(x)
axes[0].set_xticklabels(codes, rotation=45, ha='right')
axes[0].set_ylabel('Token Units')
axes[0].set_title('AI Token Supply: Minted vs Cap')
axes[0].legend()
axes[0].set_yscale('log')

# Plot 2: Fibonacci × φ² supply curve
fib_indices = range(5, 21)
supply_caps = [FIB_21[i] * PHI_SQ for i in fib_indices]
axes[1].semilogy(list(fib_indices), supply_caps, 'o-', color='purple', linewidth=2)
axes[1].set_xlabel('Fibonacci Index')
axes[1].set_ylabel('Supply Cap')
axes[1].set_title('Supply Cap Formula: Fibonacci[n] × φ²')
axes[1].grid(True, alpha=0.3)
# Mark the tokens we created
for t in factory.tokens.values():
    axes[1].axhline(y=t.supply_cap, color='red', alpha=0.2, linestyle='--')

plt.tight_layout()
plt.show()

print("\n✅ All tokens minted with phi-derived supply mechanics")
print(f"   Formula: supply_cap = Fibonacci[n] × φ² = Fib[n] × {PHI_SQ:.6f}")
print(f"   Total tokens created: {len(factory.tokens)}")
print(f"   Total mint events: {len(factory.mint_events)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5: Attempt to exceed supply cap (demonstrates enforcement)
# ═══════════════════════════════════════════════════════════════

print("SUPPLY CAP ENFORCEMENT TEST:")
print("=" * 60)

token = factory.tokens["AIVIS"]
print(f"  AIVIS supply cap: {token.supply_cap:,}")
print(f"  AIVIS minted:     {token.minted:,.0f}")
print(f"  Attempting to mint {token.supply_cap + 1:,} tokens...")

result = factory.mint("AIVIS", "attacker", token.supply_cap + 1)

if result is None:
    print(f"\n  ✅ Supply cap enforced — mint rejected")
    print(f"     Sovereign doctrine prevents inflation")
else:
    print(f"  ❌ ERROR: Supply cap was not enforced!")